# FRP — Regression of experimental FRP shear contribution (`Vf_kN`)
## Nested study-aware LOSO experiment

**Frozen dataset:** 53 specimens, 8 independent studies.

### Primary objective
Estimate experimental `Vf_kN` from pre-test physical/design variables and evaluate transfer to an entirely unseen source study.

### Feature sets fixed before regression
- **X1_full:** 17 physical/design variables.
- **X2_reduced:** 13 variables after redundancy/multicollinearity analysis.

### Leakage exclusions
`V_kN`, `FM`, `normalized_Vf_scaled`, `paper_ref`, identifiers and any variable containing experimental `Vf_kN` information are excluded from predictors.

### External validation
Leave-One-Study-Out (LOSO) by `paper_ref`. Every specimen is predicted by a model that has seen **no specimen from its source study**.

### Internal tuning
GroupKFold on training studies only. Preprocessing and tuning occur inside the outer LOSO loop.

### Models
1. Dummy median baseline
2. Linear Regression
3. Ridge
4. Lasso
5. Elastic Net
6. SVR-RBF
7. Random Forest
8. Gradient Boosting
9. XGBoost

### Primary metrics
- MAE
- RMSE
- R²

### Complementary metrics
- Median absolute error
- MAPE (interpreted cautiously because small `Vf_kN` values can inflate percentage errors)
- sMAPE

No model will be selected using a single favorable metric.

## 1. Install dependencies

In [ ]:
%pip -q install openpyxl scikit-learn xgboost

## 2. Imports and reproducibility

In [ ]:
import os, sys, json, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from xgboost import XGBRegressor
import xgboost

warnings.filterwarnings("ignore")
RANDOM_STATE = 20261001
np.random.seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn_version)
print("xgboost:", xgboost.__version__)
print("random_state:", RANDOM_STATE)

## 3. Upload/load the validated frozen workbook

In [ ]:
EXPECTED_NAMES = [
    "FRP_Dataset_v1_FINAL_VALIDATED_LOCKED.xlsx",
    "FRP_Dataset_v1_FINAL_VALIDATED_LOCKED(1).xlsx",
]
def locate_workbook():
    for base in [Path("/content"), Path(".")]:
        for name in EXPECTED_NAMES:
            p=base/name
            if p.exists(): return p
    return None

workbook=locate_workbook()
if workbook is None:
    try:
        from google.colab import files
        print("Select the frozen validated Excel workbook.")
        files.upload()
        workbook=locate_workbook()
    except Exception as e:
        raise FileNotFoundError("Upload the frozen validated workbook.") from e
if workbook is None: raise FileNotFoundError("Workbook not found.")

df=pd.read_excel(workbook, sheet_name="Dataset_v1_FROZEN")
print("Loaded:", workbook.name)
print("Shape:", df.shape)

## 4. Dataset integrity and target audit

In [ ]:
required={"row_id","paper_ref","specimen","Vf_kN","V_kN","FM","WC","bw_mm","d_mm","df_mm",
"a_d","fc_MPa","fsy_MPa","rho_s","fvy_MPa","rho_sv","tf_mm","wf_mm","sf_mm",
"ffu_MPa","Ef_GPa","rho_fv","alpha_f_deg"}
assert df.shape==(53,32), f"Expected (53,32), got {df.shape}"
assert not (required-set(df.columns)), f"Missing: {required-set(df.columns)}"
assert df.isna().sum().sum()==0
assert df["paper_ref"].nunique()==8
assert df["row_id"].is_unique
assert not df.duplicated(["paper_ref","specimen"]).any()
assert (df["Vf_kN"]>0).all()

print("✓ 53 rows x 32 columns")
print("✓ 0 missing values")
print("✓ 8 independent studies")
print("✓ unique row_id and study/specimen pairs")
print(df["Vf_kN"].describe())
print("\nStudy sizes:")
print(df.groupby("paper_ref").size())

## 5. Freeze X1 and X2 and enforce leakage guards

In [ ]:
X1=["bw_mm","d_mm","df_mm","a_d","fc_MPa","fsy_MPa","rho_s","fvy_MPa","rho_sv",
    "tf_mm","wf_mm","sf_mm","ffu_MPa","Ef_GPa","rho_fv","alpha_f_deg","WC"]
X2=["bw_mm","df_mm","a_d","fc_MPa","fsy_MPa","rho_s","fvy_MPa","rho_sv",
    "ffu_MPa","Ef_GPa","rho_fv","alpha_f_deg","WC"]
feature_sets={"X1_full":X1,"X2_reduced":X2}

FORBIDDEN={"V_kN","Vf_kN","FM","normalized_Vf_scaled","paper_ref","row_id","specimen",
           "frp_axial_rigidity_index"}
for name,cols in feature_sets.items():
    assert not (FORBIDDEN & set(cols)), f"Leakage in {name}: {FORBIDDEN & set(cols)}"
    print(name, len(cols), cols)
print("\nExcluded X1→X2:", sorted(set(X1)-set(X2)))
print("✓ Leakage guards passed.")

## 6. Preprocessing and model search spaces

In [ ]:
def preprocessor(features, scaled=True):
    cat=["WC"]; num=[c for c in features if c not in cat]
    num_pipe=StandardScaler() if scaled else "passthrough"
    return ColumnTransformer([
        ("num", num_pipe, num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)
    ], verbose_feature_names_out=False)

def model_specs(features):
    # Scaling for linear/regularized/SVR; trees use original numeric scales.
    return {
      "Dummy": (Pipeline([("prep",preprocessor(features,True)),("model",DummyRegressor(strategy="median"))]), {}),
      "Linear": (Pipeline([("prep",preprocessor(features,True)),("model",LinearRegression())]), {}),
      "Ridge": (Pipeline([("prep",preprocessor(features,True)),("model",Ridge())]),
                {"model__alpha":[0.01,0.1,1,10,100]}),
      "Lasso": (Pipeline([("prep",preprocessor(features,True)),("model",Lasso(max_iter=20000,random_state=RANDOM_STATE))]),
                {"model__alpha":[0.001,0.01,0.1,1,10]}),
      "ElasticNet": (Pipeline([("prep",preprocessor(features,True)),("model",ElasticNet(max_iter=20000,random_state=RANDOM_STATE))]),
                {"model__alpha":[0.001,0.01,0.1,1,10],"model__l1_ratio":[0.2,0.5,0.8]}),
      "SVR_RBF": (Pipeline([("prep",preprocessor(features,True)),("model",SVR(kernel="rbf"))]),
                {"model__C":[1,10,100,1000],"model__epsilon":[0.1,1,5,10],"model__gamma":["scale",0.01,0.1]}),
      "RandomForest": (Pipeline([("prep",preprocessor(features,False)),
                                ("model",RandomForestRegressor(random_state=RANDOM_STATE,n_jobs=-1))]),
                {"model__n_estimators":[300],"model__max_depth":[None,3,5],
                 "model__min_samples_leaf":[1,2,4],"model__max_features":["sqrt",0.7,1.0]}),
      "GradientBoosting": (Pipeline([("prep",preprocessor(features,False)),
                                    ("model",GradientBoostingRegressor(random_state=RANDOM_STATE))]),
                {"model__n_estimators":[50,100,200],"model__learning_rate":[0.03,0.05,0.1],
                 "model__max_depth":[1,2,3],"model__min_samples_leaf":[1,2,4]}),
      "XGBoost": (Pipeline([("prep",preprocessor(features,False)),
                           ("model",XGBRegressor(objective="reg:squarederror",random_state=RANDOM_STATE,
                                               n_jobs=-1,verbosity=0))]),
                {"model__n_estimators":[100,300],"model__max_depth":[1,2,3],
                 "model__learning_rate":[0.03,0.05,0.1],
                 "model__subsample":[0.8,1.0],"model__colsample_bytree":[0.8,1.0]})
    }
print("Models:", list(model_specs(X2).keys()))
print("Scenarios:", 2*len(model_specs(X2)))

## 7. Metrics

Global metrics are computed from the 53 concatenated out-of-study predictions. Fold-level R² can be unstable or undefined for very small held-out studies and is therefore not averaged blindly.

In [ ]:
def smape(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    den=np.abs(y)+np.abs(p)
    return 100*np.mean(np.where(den==0,0,2*np.abs(p-y)/den))

def mape(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    return 100*np.mean(np.abs((y-p)/y))

def metrics(y,p):
    return {
      "MAE":mean_absolute_error(y,p),
      "RMSE":np.sqrt(mean_squared_error(y,p)),
      "R2":r2_score(y,p) if len(y)>=2 else np.nan,
      "MedAE":median_absolute_error(y,p),
      "MAPE_pct":mape(y,p),
      "sMAPE_pct":smape(y,p)
    }

## 8. Nested LOSO experiment

In [ ]:
pred_rows=[]; fold_rows=[]; hp_rows=[]
studies=sorted(df["paper_ref"].unique())
y_all=df["Vf_kN"].astype(float)

for fs_name,features in feature_sets.items():
    X=df[features].copy()
    specs=model_specs(features)
    for model_name,(pipe,param_grid) in specs.items():
        print(f"[{fs_name}] {model_name}")
        for heldout in studies:
            test_mask=df["paper_ref"].eq(heldout)
            train_mask=~test_mask
            Xtr,Xte=X.loc[train_mask],X.loc[test_mask]
            ytr,yte=y_all.loc[train_mask],y_all.loc[test_mask]
            gtr=df.loc[train_mask,"paper_ref"]

            if param_grid:
                n_groups=gtr.nunique()
                inner=GroupKFold(n_splits=min(5,n_groups))
                search=GridSearchCV(pipe,param_grid,scoring="neg_mean_absolute_error",
                                    cv=inner,n_jobs=-1,refit=True,error_score="raise")
                search.fit(Xtr,ytr,groups=gtr)
                fitted=search.best_estimator_
                best_params=search.best_params_
                best_cv_mae=-search.best_score_
            else:
                fitted=pipe.fit(Xtr,ytr)
                best_params={}
                best_cv_mae=np.nan

            pred=fitted.predict(Xte)
            fm=metrics(yte,pred)
            fold_rows.append({"feature_set":fs_name,"model":model_name,"paper_ref":heldout,
                              "N_test":len(yte),**fm})
            hp_rows.append({"feature_set":fs_name,"model":model_name,"paper_ref":heldout,
                            "best_params":json.dumps(best_params,sort_keys=True),
                            "inner_CV_MAE":best_cv_mae})

            for idx,pr in zip(df.index[test_mask],pred):
                pred_rows.append({
                    "feature_set":fs_name,"model":model_name,
                    "row_id":df.loc[idx,"row_id"],"paper_ref":df.loc[idx,"paper_ref"],
                    "specimen":df.loc[idx,"specimen"],
                    "Vf_real":float(df.loc[idx,"Vf_kN"]),"Vf_pred":float(pr)
                })

predictions=pd.DataFrame(pred_rows)
fold_results=pd.DataFrame(fold_rows)
hyperparameters=pd.DataFrame(hp_rows)

n_models=len(model_specs(X2))
print("\nPredictions:",len(predictions),"Expected:",53*2*n_models)
print("Fold results:",len(fold_results),"Expected:",8*2*n_models)
print("Hyperparameter rows:",len(hyperparameters),"Expected:",8*2*n_models)
assert len(predictions)==53*2*n_models
assert len(fold_results)==8*2*n_models
assert not predictions.duplicated(["feature_set","model","row_id"]).any()
print("✓ Nested LOSO completed.")

## 9. Immediate backup

In [ ]:
BACKUP=Path("/content/regression_backup"); BACKUP.mkdir(exist_ok=True)
predictions.to_csv(BACKUP/"regression_predictions.csv",index=False)
fold_results.to_csv(BACKUP/"regression_by_fold.csv",index=False)
hyperparameters.to_csv(BACKUP/"best_hyperparameters.csv",index=False)
print("Backup written:",BACKUP)

## 10. Global out-of-study metrics and ranking

In [ ]:
summary=[]
for (fs,m),g in predictions.groupby(["feature_set","model"]):
    met=metrics(g["Vf_real"],g["Vf_pred"])
    summary.append({"feature_set":fs,"model":m,**met,"N":len(g)})
summary=pd.DataFrame(summary)

# Rank using the three pre-specified primary metrics.
summary["rank_MAE"]=summary["MAE"].rank(ascending=True,method="min")
summary["rank_RMSE"]=summary["RMSE"].rank(ascending=True,method="min")
summary["rank_R2"]=summary["R2"].rank(ascending=False,method="min")
summary["aggregate_primary_rank"]=summary[["rank_MAE","rank_RMSE","rank_R2"]].mean(axis=1)
summary=summary.sort_values(["aggregate_primary_rank","MAE","RMSE"]).reset_index(drop=True)
display(summary)
print("\nPrimary by pre-specified aggregate rank:")
display(summary.head(1))

## 11. Per-study generalization of the leading solutions

In [ ]:
leaders=summary.head(5)[["feature_set","model"]]
leader_folds=fold_results.merge(leaders,on=["feature_set","model"],how="inner")
display(leader_folds.sort_values(["feature_set","model","paper_ref"]))

## 12. Observed vs predicted plots for all 18 scenarios

In [ ]:
FIG=Path("/content/regression_figures"); FIG.mkdir(exist_ok=True)
for (fs,m),g in predictions.groupby(["feature_set","model"]):
    plt.figure(figsize=(6,6))
    plt.scatter(g["Vf_real"],g["Vf_pred"],s=50,alpha=.8)
    lo=min(g["Vf_real"].min(),g["Vf_pred"].min())
    hi=max(g["Vf_real"].max(),g["Vf_pred"].max())
    plt.plot([lo,hi],[lo,hi],"--")
    plt.xlabel("Observed Vf_kN")
    plt.ylabel("Out-of-study predicted Vf_kN")
    plt.title(f"{fs} — {m}")
    plt.tight_layout()
    plt.savefig(FIG/f"{fs}__{m}.png",dpi=300,bbox_inches="tight")
    plt.close()
print("Saved figures:",len(list(FIG.glob("*.png"))))

## 13. Detailed residual audit for the primary model

In [ ]:
best_fs=summary.iloc[0]["feature_set"]; best_model=summary.iloc[0]["model"]
best=predictions[(predictions.feature_set==best_fs)&(predictions.model==best_model)].copy()
best["residual"]=best["Vf_real"]-best["Vf_pred"]
best["abs_error"]=best["residual"].abs()
best["APE_pct"]=100*best["abs_error"]/best["Vf_real"]
print("Primary:",best_fs,best_model)
display(best.sort_values("abs_error",ascending=False).head(15))

print("\nErrors by held-out study:")
by_study=best.groupby("paper_ref").apply(
    lambda g: pd.Series(metrics(g["Vf_real"],g["Vf_pred"])),
    include_groups=False
)
display(by_study)

## 14. X1 vs X2 paired comparison by model

In [ ]:
wide=summary.pivot(index="model",columns="feature_set",
                   values=["MAE","RMSE","R2","MedAE","MAPE_pct","sMAPE_pct"])
display(wide)

## 15. Export complete reproducibility package

In [ ]:
OUT=Path("/content/regression_results"); OUT.mkdir(exist_ok=True)
predictions.to_csv(OUT/"regression_predictions.csv",index=False)
fold_results.to_csv(OUT/"regression_by_fold.csv",index=False)
hyperparameters.to_csv(OUT/"best_hyperparameters.csv",index=False)
summary.to_csv(OUT/"regression_summary.csv",index=False)
best.to_csv(OUT/"primary_model_predictions_and_residuals.csv",index=False)
by_study.to_csv(OUT/"primary_model_by_study.csv")

study_dist=df.groupby("paper_ref").agg(N=("row_id","size"),
                                      Vf_mean=("Vf_kN","mean"),
                                      Vf_median=("Vf_kN","median"),
                                      Vf_min=("Vf_kN","min"),
                                      Vf_max=("Vf_kN","max")).reset_index()
study_dist.to_csv(OUT/"Vf_distribution_by_study.csv",index=False)

primary={"feature_set":best_fs,"model":best_model,
         "selection":"Mean rank of global OOS MAE, RMSE and R2",
         "random_state":RANDOM_STATE,
         "outer_validation":"Leave-One-Study-Out by paper_ref",
         "inner_validation":"GroupKFold on training studies only"}
with open(OUT/"primary_solution.json","w") as f: json.dump(primary,f,indent=2)

versions={"python":sys.version,"platform":platform.platform(),"pandas":pd.__version__,
          "numpy":np.__version__,"scikit_learn":sklearn_version,
          "xgboost":xgboost.__version__,"random_state":RANDOM_STATE}
with open(OUT/"environment_versions.json","w") as f: json.dump(versions,f,indent=2)

# Copy figures
import shutil
for p in FIG.glob("*.png"): shutil.copy2(p,OUT/p.name)

print("All outputs written to:",OUT)
for p in sorted(OUT.iterdir()): print("-",p.name)

## 16. Create and download ZIP

In [ ]:
import shutil
zip_path=shutil.make_archive("/content/FRP_regression_results","zip","/content/regression_results")
print("Created:",zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually from the Colab Files panel.")

# Interpretation guardrails

1. **Do not use random train/test splits** as the principal validation: study-aware LOSO is the external test.
2. **Do not use `V_kN`, `FM`, `normalized_Vf_scaled` or any target-derived feature** to predict `Vf_kN`.
3. Global OOS metrics are calculated after concatenating the 53 LOSO predictions.
4. Fold-level R² is not averaged blindly, especially for tiny held-out studies.
5. MAPE is complementary only because `Vf_kN` includes very small positive values.
6. With N=53, differences among models should be interpreted together with per-study stability and residual patterns.
7. Hyperparameters are selected independently inside each outer training fold; there is no single global hyperparameter set during unbiased evaluation.
8. Do not add models or tune the search space after seeing results merely to improve the reported score.